# 01 · KVPI – Cálculo y Análisis Multianual
### KiwiTrackingIA · Lote "El Abrojito" · 2019–2025

**Descripción:**  
Pipeline completo de cálculo del KVPI (*Kiwi Vegetation Productivity Index*) a partir de imágenes Sentinel-2 en Google Earth Engine.  
Genera la serie histórica por planta (2019–2025) y calcula indicadores de potencial estructural y estabilidad interanual.

**Inputs requeridos:**
- `datos/muestreo_sistematico.csv` — coordenadas (X, Y) y ID de cada planta
- Acceso a Google Earth Engine (cuenta autenticada)

**Outputs generados:**
- `datos/KVPI_serie_2019_2025.csv` — serie histórica exportada desde GEE
- Mapas estáticos y mapa Folium interactivo de potencial y estabilidad


## 1 · Librerías y configuración

In [ ]:
# ============================================================
# KiwiTrackingIA – 01_KVPI_Calculo.ipynb
# Autor: Darío Nicolás Sánchez Leguizamón
# Fecha: 2026-03-30
# Licencia: MIT
# Repositorio: https://github.com/sdarionicolas-boop/KiwiTrackingIA
# ============================================================

import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium import plugins

pd.set_option('display.max_columns', None)
plt.style.use('default')

print("✅ Librerías cargadas correctamente")

## 2 · Inicialización de Google Earth Engine

In [ ]:
# Autenticar y conectar (ejecutar solo una vez por sesión)
ee.Authenticate()
ee.Initialize(project='TU_PROYECTO_GEE')  # ← reemplazar con tu project ID

# Verificación rápida
test = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').first()
print("✅ Google Earth Engine inicializado")
print("🛰️  Acceso a Sentinel-2 OK:", test is not None)

## 3 · Carga de puntos productivos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

csv_puntos = "/content/drive/MyDrive/KiwiTrackingIA/muestreo_sistematico.csv"
df_pts = pd.read_csv(csv_puntos)

print("🌱 Puntos productivos cargados:", len(df_pts))
display(df_pts.head())

In [ ]:
# Control espacial de puntos
plt.figure(figsize=(6, 6))
plt.scatter(df_pts['X'], df_pts['Y'], s=30, alpha=0.7)
plt.xlabel("Longitud"); plt.ylabel("Latitud")
plt.title("Puntos productivos – El Abrojito")
plt.grid(alpha=0.3); plt.axis('equal')
plt.tight_layout()
plt.show()

print("📐 Rango espacial:")
print("Longitud:", df_pts['X'].min(), "→", df_pts['X'].max())
print("Latitud :", df_pts['Y'].min(), "→", df_pts['Y'].max())

## 4 · Definición de campañas agrícolas (junio–junio)

In [ ]:
campanias = {
    "2019_2020": ("2019-06-01", "2020-06-30"),
    "2020_2021": ("2020-06-01", "2021-06-30"),
    "2021_2022": ("2021-06-01", "2022-06-30"),
    "2022_2023": ("2022-06-01", "2023-06-30"),
    "2023_2024": ("2023-06-01", "2024-06-30"),
    "2024_2025": ("2024-06-01", "2025-06-30"),
}

print("📅 Campañas definidas:")
for k, v in campanias.items():
    print(f"  {k}: {v[0]} → {v[1]}")

## 5 · Pipeline GEE: carga Sentinel-2, índices espectrales y KVPI

In [ ]:
def mask_s2_clouds(image):
    """Máscara de nubes para Sentinel-2 SR usando banda QA60."""
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )
    return (
        image.updateMask(mask)
             .divide(10000)
             .copyProperties(image, ["system:time_start"])
    )


def load_s2_collection(start_date, end_date, roi):
    """Carga colección Sentinel-2 SR, filtra nubes < 20% y recorta al ROI."""
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start_date, end_date)
        .filterBounds(roi)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .map(mask_s2_clouds)
    )


def add_spectral_indices(image):
    """
    Calcula índices espectrales relevantes para kiwi:
      NDVI: vigor general (B8/B4)
      NDRE: actividad fotosintética (B8/B5)
      NDMI: condición hídrica (B8/B11)
      EVI:  vigor mejorado con corrección atmosférica
    """
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndre = image.normalizedDifference(['B8', 'B5']).rename('NDRE')
    ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')
    evi  = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {'NIR': image.select('B8'), 'RED': image.select('B4'), 'BLUE': image.select('B2')}
    ).rename('EVI')
    return image.addBands([ndvi, ndre, ndmi, evi])


def compute_kvpi(image):
    """KVPI = promedio simple de NDVI, NDRE, EVI, NDMI (todos normalizados en [-1,1])."""
    indices = image.select(['NDVI', 'NDRE', 'EVI', 'NDMI'])
    kvpi = indices.reduce(ee.Reducer.mean()).rename('KVPI')
    return kvpi.copyProperties(image, ["system:time_start"])


def kvpi_for_campaign(start_date, end_date, roi):
    """Retorna la imagen KVPI mediana de una campaña completa."""
    s2       = load_s2_collection(start_date, end_date, roi)
    s2_idx   = s2.map(add_spectral_indices)
    kvpi_col = s2_idx.map(compute_kvpi)
    return kvpi_col.median()

print("✅ Funciones GEE definidas")

## 6 · Construcción del ROI y cálculo KVPI por campaña

In [ ]:
# Crear FeatureCollection de plantas en GEE
features = []
for _, r in df_pts.iterrows():
    geom  = ee.Geometry.Point([float(r['X']), float(r['Y'])])
    props = {'id': int(r['id'])}
    if 'row_index' in df_pts.columns:
        props['row'] = int(r['row_index'])
    if 'col_index' in df_pts.columns:
        props['col'] = int(r['col_index'])
    features.append(ee.Feature(geom, props))

fc_plantas = ee.FeatureCollection(features)
roi        = fc_plantas.geometry().buffer(50)

print("🌱 fc_plantas creada:", fc_plantas.size().getInfo(), "puntos")
print("📐 ROI definido (buffer 50 m sobre bounding box)")

In [ ]:
# Calcular KVPI para cada campaña
kvpi_images = {}
for camp, (start, end) in campanias.items():
    print(f"🛰️  Procesando campaña {camp}...")
    img = kvpi_for_campaign(start, end, roi).rename('KVPI')
    kvpi_images[camp] = img

print("\n✅ KVPI calculado para todas las campañas:")
print(list(kvpi_images.keys()))

## 7 · Muestreo KVPI por planta y exportación a Drive

In [ ]:
def sample_kvpi_by_plants(kvpi_image, fc_plants, campaign):
    """Extrae valores KVPI por punto planta y agrega el nombre de campaña."""
    sampled = kvpi_image.sampleRegions(
        collection=fc_plants,
        properties=['id', 'row', 'col'],
        scale=10,
        geometries=True
    )
    return sampled.map(lambda f: f.set('campaña', campaign))


samples_all = []
for camp, img in kvpi_images.items():
    print(f"🌱 Muestreando – campaña {camp}")
    samples_all.append(sample_kvpi_by_plants(img, fc_plantas, camp))

fc_kvpi_all = ee.FeatureCollection(samples_all).flatten()
print("\n✅ Muestreo completo. Total registros:", fc_kvpi_all.size().getInfo())

In [ ]:
# Exportar a Drive (tarea asíncrona de GEE)
export_task = ee.batch.Export.table.toDrive(
    collection=fc_kvpi_all,
    description='KVPI_Serie_2019_2025_Plantas',
    folder='KiwiTrackingIA',
    fileNamePrefix='KVPI_serie_2019_2025',
    fileFormat='CSV'
)
export_task.start()

print("🚀 Exportación iniciada.")
print("👉 Revisá el panel de tareas de GEE (Tasks) para monitorear el progreso.")
print("   Una vez completada, el archivo quedará en Drive/KiwiTrackingIA/KVPI_serie_2019_2025.csv")

## 8 · Análisis multianual: construcción de indicadores por planta

In [ ]:
# Cargar CSV exportado por GEE
csv_kvpi = "/content/drive/MyDrive/KiwiTrackingIA/KVPI_serie_2019_2025.csv"
df_long = pd.read_csv(csv_kvpi)

print("📋 Serie cargada:", df_long.shape)
display(df_long.head())

In [ ]:
# Matriz temporal: filas = plantas, columnas = campañas
df_matrix = (
    df_long
    .pivot_table(index='id', columns='campaña', values='KVPI')
    .sort_index()
)
print("📊 Matriz temporal KVPI (primeras filas):")
display(df_matrix.head())

In [ ]:
# Indicadores por planta
df_indicadores = pd.DataFrame(index=df_matrix.index)
df_indicadores['KVPI_mean']       = df_matrix.mean(axis=1)
df_indicadores['KVPI_std']        = df_matrix.std(axis=1)
df_indicadores['KVPI_cv']         = df_indicadores['KVPI_std'] / df_indicadores['KVPI_mean']
df_indicadores['KVPI_estabilidad'] = df_indicadores['KVPI_mean'] / df_indicadores['KVPI_std']

# Persistencia de clases
df_classes = df_matrix.apply(
    lambda col: pd.qcut(col, q=3, labels=['Bajo', 'Medio', 'Alto']),
    axis=0
)
df_indicadores['años_alto']  = (df_classes == 'Alto').sum(axis=1)
df_indicadores['años_medio'] = (df_classes == 'Medio').sum(axis=1)
df_indicadores['años_bajo']  = (df_classes == 'Bajo').sum(axis=1)

print("✅ Indicadores calculados para", len(df_indicadores), "plantas")
display(df_indicadores.head())

## 9 · Clasificación operativa de plantas

In [ ]:
# Umbrales por cuantiles (p33/p66 para potencial, p33/p66 para estabilidad)
q_pot = df_indicadores['KVPI_mean'].quantile([0.33, 0.66])
q_est = df_indicadores['KVPI_estabilidad'].quantile([0.33, 0.66])

def clasificar_planta(row):
    if row['KVPI_mean'] <= q_pot[0.33] and row['KVPI_estabilidad'] >= q_est[0.66]:
        return 'Intervenir'
    elif row['KVPI_mean'] >= q_pot[0.66] and row['KVPI_estabilidad'] >= q_est[0.66]:
        return 'Mantener'
    else:
        return 'Observar'

df_indicadores['clase_manejo'] = df_indicadores.apply(clasificar_planta, axis=1)

print("📊 Distribución de clases de manejo:")
print(df_indicadores['clase_manejo'].value_counts())

## 10 · Visualización espacial

In [ ]:
# Unir coordenadas e indicadores
df_geo = df_pts.merge(df_indicadores.reset_index(), on='id', how='inner')
df_geo['KVPI_estabilidad'] = df_geo['KVPI_estabilidad'].replace([np.inf, -np.inf], np.nan)
df_geo = df_geo.dropna(subset=['KVPI_estabilidad'])

def robust_minmax(s, lo=0.02, hi=0.98):
    return s.quantile(lo), s.quantile(hi)

# Mapa estático: potencial estructural
vmin, vmax = robust_minmax(df_geo['KVPI_mean'])
plt.figure(figsize=(7, 7))
sc = plt.scatter(df_geo['X'], df_geo['Y'],
                 c=df_geo['KVPI_mean'], s=45, vmin=vmin, vmax=vmax)
plt.colorbar(sc, label='KVPI_mean')
plt.title("KVPI medio multianual (Potencial estructural)\nEl Abrojito · 2019–2025")
plt.xlabel("Longitud"); plt.ylabel("Latitud")
plt.grid(alpha=0.3); plt.axis('equal')
plt.tight_layout(); plt.show()

# Mapa estático: estabilidad
vmin, vmax = robust_minmax(df_geo['KVPI_estabilidad'])
plt.figure(figsize=(7, 7))
sc = plt.scatter(df_geo['X'], df_geo['Y'],
                 c=df_geo['KVPI_estabilidad'], s=45, vmin=vmin, vmax=vmax, cmap='RdYlGn')
plt.colorbar(sc, label='KVPI_estabilidad (mean/std)')
plt.title("Estabilidad multianual (KVPI mean/std)\nEl Abrojito · 2019–2025")
plt.xlabel("Longitud"); plt.ylabel("Latitud")
plt.grid(alpha=0.3); plt.axis('equal')
plt.tight_layout(); plt.show()

In [ ]:
# Mapa interactivo Folium (modo productor)
df_geo['clase_potencial'] = pd.qcut(df_geo['KVPI_mean'], q=3, labels=['Bajo', 'Medio', 'Alto'])
color_map = {'Bajo': 'red', 'Medio': 'orange', 'Alto': 'green'}

m = folium.Map(
    location=[df_geo['Y'].mean(), df_geo['X'].mean()],
    zoom_start=17,
    tiles="OpenStreetMap",
    control_scale=True
)

for _, r in df_geo.iterrows():
    popup_html = f"""
    <b>Planta ID:</b> {int(r['id'])}<br>
    <b>Potencial (KVPI_mean):</b> {r['KVPI_mean']:.3f}<br>
    <b>Variabilidad (CV):</b> {r['KVPI_cv']:.3f}<br>
    <b>Estabilidad (mean/std):</b> {r['KVPI_estabilidad']:.2f}<br>
    <hr>
    <b>Persistencia 2019–2025:</b><br>
    Alto: {int(r.get('años_alto', 0))} años &nbsp;
    Medio: {int(r.get('años_medio', 0))} años &nbsp;
    Bajo: {int(r.get('años_bajo', 0))} años
    """
    folium.CircleMarker(
        location=[r['Y'], r['X']],
        radius=5,
        color=color_map[str(r['clase_potencial'])],
        fill=True, fill_opacity=0.85,
        tooltip=f"Planta {int(r['id'])} | Potencial: {r['clase_potencial']}",
        popup=folium.Popup(popup_html, max_width=320)
    ).add_to(m)

# Capas por potencial
for clase, fg_name in [('Alto', '🟢 Alto'), ('Medio', '🟡 Medio'), ('Bajo', '🔴 Bajo')]:
    fg = folium.FeatureGroup(name=f'Potencial {fg_name}')
    for _, r in df_geo[df_geo['clase_potencial'] == clase].iterrows():
        folium.CircleMarker([r['Y'], r['X']], radius=4,
                            color=color_map[clase], fill=True,
                            fill_color=color_map[clase], fill_opacity=0.6).add_to(fg)
    fg.add_to(m)

# Heatmap
heat_data = [[r['Y'], r['X'], r['KVPI_mean']] for _, r in df_geo.iterrows()]
plugins.HeatMap(heat_data, radius=15, blur=10,
                gradient={0.2:'red',0.5:'orange',0.8:'yellow',1:'green'},
                name='🔥 Heatmap KVPI').add_to(m)

folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google', name='🛰️ Satélite'
).add_to(m)

plugins.MiniMap(toggle_display=True, position='bottomright', width=150, height=150).add_to(m)

legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;
            padding:12px 14px;border:2px solid #444;border-radius:8px;
            font-size:13px;line-height:1.5;box-shadow:2px 2px 6px rgba(0,0,0,0.3);">
<b>Mapa KVPI – ¿Qué estoy viendo?</b><br>
<span style="font-size:12px;">Cada punto = una planta de kiwi.<br>
Color = potencial productivo relativo (2019–2025)</span>
<hr style="margin:6px 0;">
<span style="color:green;font-weight:bold;">● Verde:</span> Potencial alto<br>
<span style="color:orange;font-weight:bold;">● Naranja:</span> Potencial medio<br>
<span style="color:red;font-weight:bold;">● Rojo:</span> Potencial bajo
<hr style="margin:6px 0;">
<span style="font-size:11px;">🖱️ Hover → info rápida | Click → detalles</span>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(position='topright', collapsed=False).add_to(m)

# Guardar HTML
out_html = "/content/drive/MyDrive/KiwiTrackingIA/KVPI_Mapa_Productor_ElAbrojito.html"
m.save(out_html)
print("✅ Mapa guardado en:", out_html)
m